**PROYECTO:
Modelado espacial para estimar la susceptibilidad a inundaciones usando algoritmos de aprendizaje automatico con datos geomorfologicos**

Autor: Sebastian Velosa Bohorquez. Curso: Percepción Remota Avanzada.


**NOTEBOOK 02/07**

**TITULO NOTEBOOK:** ARMONIZACION ESPACIAL DE DATOS DEM/DTM


**Objetivo**

Este notebook presenta un procedimiento para la armonizacion espacial de datos raster obtenidos del DEM COPERNICUS_GLO30,ANADEM y FABDEM para el area de estudio de la cuenca baja del rio sinu.

**Herramientas:** Google Earth Engine Code Editor + Google Colab.

**Datos:** COPERNICUS GLO 30 DEM, FABDEM, ANADEM exportados de GEE. SHAPE AREA ZONA DE ESTUDIO

**Contenido:**

1. Instalacion de Librerias
2. Configurar las rutas de los archivos
3. Leer y validar el archivo ".gpkg" del area de estudio **AOI**
4. Creación de una cuadrícula estandar
5. Mascara espacial AOI
6. Definir la función de alineación
7. Verificación independiente de los GeoTIFF
8. Manifiesto de reproducibilidad

**Resultado:**
Se obtienen tres DEM en formato "`.tiff`" . Alineados a una misma grilla y sistema de referencia.



*   ANADEM_Sinu_alineado.tif"
*   COPERNICUS_GLO30_Sinu_alineado.tif"
*   FABDEM_Sinu_alineado.tif"




## 1. Instalar e importar librerias

In [ ]:
%pip install -q geopandas rasterio shapely pyproj pandas

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import platform
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import shapely
import pyproj

from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask

Versiones trabajadas en este notebook:

In [ ]:
VERSIONES = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "geopandas": gpd.__version__,
    "rasterio": rasterio.__version__,
    "shapely": shapely.__version__,
    "pyproj": pyproj.__version__,
}

print(json.dumps(VERSIONES, indent=2, ensure_ascii=False))

{
  "python": "3.13.18",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "geopandas": "1.2.0",
  "rasterio": "1.5.2",
  "shapely": "2.2.0",
  "pyproj": "3.8.0"
}


##2. Configurar las rutas de los archivos

In [ ]:
# De Google Drive:
from google.colab import drive
drive.mount("/content/drive")

RUTA_AOI = Path(
    "/content/drive/MyDrive/Sinu_Flood_Susceptibility/02_Data_raw/AOI/Perimetro_AOI_SINU.gpkg"
)

CARPETA_DEM = Path(
    "/content/drive/MyDrive/Sinu_Flood_Susceptibility/02_Data_raw/DEM"
)

CARPETA_SALIDA = Path(
    "/content/drive/MyDrive/Sinu_Flood_Susceptibility/02_Data_raw/DEM_alineados"
)



# Parámetros espaciales del proyecto

# Sistema de referencia horizontal
CRS_TRABAJO = "EPSG:31972" #SIRGAS 2000 / UTM 18N

# Cuadrícula común
RESOLUCION = 30.0  #Tamaño de celda
NODATA_SALIDA = -9999.0

OUTPUT_DTYPE = "float32"
RESAMPLING = "bilinear"

# Criterios espaciales
COMMON_GRID = True
COMMON_VALID_MASK = True

# Referencia vertical:
# alturas ortométricas referidas a EGM2008
VERTICAL_DATUM = "EGM2008"

# No aplicar corrección vertical automática.
VERTICAL_TRANSFORMATION = None

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

assert RUTA_AOI.exists(), f"No existe el AOI: {RUTA_AOI}"
assert CARPETA_DEM.exists(), f"No existe la carpeta: {CARPETA_DEM}"

print("Configuración de rutas correcta.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Configuración de rutas correcta.


##3. Leer y validar el archivo ".gpkg" del area de estudio **AOI**

In [ ]:
assert RUTA_AOI.suffix.lower() == ".gpkg", (
    "Se esperaba un archivo GeoPackage (.gpkg)."
)

capas_aoi = gpd.list_layers(RUTA_AOI)
print(capas_aoi)

                 name geometry_type
0  Perimetro_AOI_SINU  MultiPolygon


In [ ]:
if len(capas_aoi) != 1:
    raise ValueError(
        "El GeoPackage tiene varias capas. "
        "Selecciona explícitamente la capa del AOI."
    )

NOMBRE_CAPA_AOI = capas_aoi.iloc[0]["name"]

aoi = gpd.read_file(
    RUTA_AOI,
    layer=NOMBRE_CAPA_AOI
)

if aoi.empty:
    raise ValueError("El AOI está vacío.")

if aoi.crs is None:
    raise ValueError(
        "El AOI no tiene CRS definido. "
        "No se debe asignar uno por suposición."
    )

if aoi.geometry.isna().any() or aoi.geometry.is_empty.any():
    raise ValueError("El AOI contiene geometrías nulas o vacías.")

if not aoi.geometry.is_valid.all():
    raise ValueError(
        "El AOI tiene geometrías inválidas. "
        "Revísalas y corrígelas antes de continuar."
    )

if not aoi.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("El AOI debe contener polígonos.")

aoi = aoi.to_crs(CRS_TRABAJO)
geometrias_aoi = [g.__geo_interface__ for g in aoi.geometry]

print("AOI validado.")
print("CRS:", aoi.crs)
print("Número de geometrías:", len(aoi))
print("Área aproximada (km²):", aoi.geometry.area.sum() / 1e6)

AOI validado.
CRS: EPSG:31972
Número de geometrías: 1
Área aproximada (km²): 5867.71254197931


##4. Creación de una cuadrícula común estandar

Se crea una cuadricula rectangular con pixeles de 30m x 30m sobre el area de estudio. Esta grilla nos servira de guia alinear todos los DEM y facilitar la comparación

In [ ]:
xmin, ymin, xmax, ymax = aoi.total_bounds

xmin_grid = math.floor(xmin / RESOLUCION) * RESOLUCION
ymin_grid = math.floor(ymin / RESOLUCION) * RESOLUCION
xmax_grid = math.ceil(xmax / RESOLUCION) * RESOLUCION
ymax_grid = math.ceil(ymax / RESOLUCION) * RESOLUCION

ancho = int(round((xmax_grid - xmin_grid) / RESOLUCION))
alto = int(round((ymax_grid - ymin_grid) / RESOLUCION))

transform_grid = from_origin(
    xmin_grid,
    ymax_grid,
    RESOLUCION,
    RESOLUCION
)

assert ancho > 0 and alto > 0

print("CRS:", CRS_TRABAJO)
print("Resolución:", RESOLUCION, "m")
print("Dimensiones:", ancho, "columnas x", alto, "filas")
print("Transformación:", transform_grid)
print(
    "Extensión:",
    (xmin_grid, ymin_grid, xmax_grid, ymax_grid)
)

CRS: EPSG:31972
Resolución: 30.0 m
Dimensiones: 3103 columnas x 3848 filas
Transformación: | 30.00, 0.00, 364080.00|
| 0.00,-30.00, 1044510.00|
| 0.00, 0.00, 1.00|
Extensión: (364080.0, 929070.0, 457170.0, 1044510.0)


##5. Construir la máscara espacial con el poligono del area de estudio

La cuadrícula es rectangular; por lo que no implica que todos los píxeles estén dentro del AOI.  Se aplica una  máscara en la siguiente celda.

In [ ]:
mascara_fuera_aoi = geometry_mask(
    geometrias_aoi,
    out_shape=(alto, ancho),
    transform=transform_grid,
    all_touched=False,
    invert=False
)

pixeles_dentro = int((~mascara_fuera_aoi).sum())
pixeles_fuera = int(mascara_fuera_aoi.sum())

print("Píxeles dentro del AOI:", pixeles_dentro)
print("Píxeles fuera del AOI:", pixeles_fuera)

assert pixeles_dentro > 0

Píxeles dentro del AOI: 6519713
Píxeles fuera del AOI: 5420631


##6. Definir la función de alineación

Esta funcion comprueba los archivos de entrada, respeta el NoData declarado y utiliza siempre la misma cuadrícula. Además, aplica la máscara del AOI a los resultados



In [ ]:
def alinear_dem_con_fallback(
    ruta_entrada,
    ruta_salida,
    crs_destino,
    transform_destino,
    ancho_destino,
    alto_destino,
    mascara_fuera,
    nodata_salida=-9999.0,
    metodo=Resampling.bilinear,
    nodata_entrada_fallback=-9999.0
):
    ruta_entrada = Path(ruta_entrada)
    ruta_salida = Path(ruta_salida)

    if not ruta_entrada.exists():
        raise FileNotFoundError(ruta_entrada)

    if ruta_entrada.resolve() == ruta_salida.resolve():
        raise ValueError("La salida no puede sobrescribir la entrada.")

    with rasterio.open(ruta_entrada) as src:
        if src.crs is None:
            raise ValueError(f"El DEM no tiene CRS definido: {ruta_entrada.name}")
        if src.count != 1:
            raise ValueError(f"Se esperaba un DEM de una banda: {ruta_entrada.name}")

        # Si el DEM no declara NoData, usamos el valor de fallback indicado (-9999.0 para ANADEM)
        nodata_in = src.nodata if src.nodata is not None else nodata_entrada_fallback
        if src.nodata is None:
            print(f"[Aviso] {ruta_entrada.name} no declara NoData. Se usará {nodata_entrada_fallback} como NoData de entrada.")

        datos = src.read(1, masked=True)
        # Asegurar que se reconozca el valor de NoData si se usó el fallback
        if src.nodata is None:
            datos = np.ma.masked_equal(datos, nodata_entrada_fallback)

        fuente = datos.astype(np.float32).filled(np.nan)
        validos = np.isfinite(fuente)

        if not validos.any():
            raise ValueError(f"No hay elevaciones válidas en {ruta_entrada.name}")

        fuente[~validos] = np.nan
        destino = np.full((alto_destino, ancho_destino), nodata_salida, dtype=np.float32)

        reproject(
            source=fuente,
            destination=destino,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=np.nan,
            dst_transform=transform_destino,
            dst_crs=crs_destino,
            dst_nodata=nodata_salida,
            resampling=metodo,
            init_dest_nodata=True
        )
 # Aplicar máscara del área de estudio.
        validos_destino = np.isfinite(destino) & (destino != nodata_salida)
        destino[mascara_fuera] = nodata_salida
        destino[~validos_destino] = nodata_salida

        perfil = {
            "driver": "GTiff",
            "height": alto_destino,
            "width": ancho_destino,
            "count": 1,
            "dtype": "float32",
            "crs": crs_destino,
            "transform": transform_destino,
            "nodata": nodata_salida,
            "compress": "deflate"
        }

        ruta_salida.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(ruta_salida, "w", **perfil) as dst:
            dst.write(destino, 1)

    return {
        "entrada": ruta_entrada.name,
        "salida": ruta_salida.name,
        "metodo": metodo.name,
        "pixeles_validos": int(((destino != nodata_salida) & np.isfinite(destino)).sum()),
        "pixeles_nodata": int(((destino == nodata_salida) | ~np.isfinite(destino)).sum())
    }

entradas = sorted(CARPETA_DEM.glob("*.tif"))

if not entradas:
    raise FileNotFoundError(f"No se encontraron archivos .tif en {CARPETA_DEM}")

resultados = []

for entrada in entradas:
    salida = CARPETA_SALIDA / f"{entrada.stem}_alineado.tif"

    if salida.exists():
        salida.unlink()

    resultado = alinear_dem_con_fallback(
        ruta_entrada=entrada,
        ruta_salida=salida,
        crs_destino=CRS_TRABAJO,
        transform_destino=transform_grid,
        ancho_destino=ancho,
        alto_destino=alto,
        mascara_fuera=mascara_fuera_aoi,
        nodata_salida=NODATA_SALIDA,
        metodo=Resampling.bilinear,
        nodata_entrada_fallback=-9999.0
    )

    resultados.append(resultado)

informe = pd.DataFrame(resultados)
display(informe)

[Aviso] ANADEM_Sinu.tif no declara NoData. Se usará -9999.0 como NoData de entrada.
[Aviso] Copernicus_GLO30_Sinu.tif no declara NoData. Se usará -9999.0 como NoData de entrada.
[Aviso] FABDEM_Sinu_V12.tif no declara NoData. Se usará -9999.0 como NoData de entrada.


,entrada,salida,metodo,pixeles_validos,pixeles_nodata
0,ANADEM_Sinu.tif,ANADEM_Sinu_alineado.tif,bilinear,6518932,5421412
1,Copernicus_GLO30_Sinu.tif,Copernicus_GLO30_Sinu_alineado.tif,bilinear,6519713,5420631
2,FABDEM_Sinu_V12.tif,FABDEM_Sinu_V12_alineado.tif,bilinear,6519713,5420631


##7.Verificación independiente de los GeoTIFF

In [ ]:
def verificar_dem(ruta, referencia_transform, referencia_crs,
                  referencia_ancho, referencia_alto,
                  mascara_fuera, nodata_esperado):

    with rasterio.open(ruta) as src:
        datos = src.read(1)

        checks = {
            "crs_correcto": src.crs == rasterio.crs.CRS.from_user_input(
                referencia_crs
            ),
            "transform_correcto": src.transform == referencia_transform,
            "ancho_correcto": src.width == referencia_ancho,
            "alto_correcto": src.height == referencia_alto,
            "resolucion_correcta": np.allclose(
                np.abs(src.res), [RESOLUCION, RESOLUCION]
            ),
            "nodata_correcto": src.nodata == nodata_esperado,
            "una_banda": src.count == 1,
            "dtype_float32": src.dtypes[0] == "float32",
            "sin_infinitos_validos": not np.isinf(
                datos[datos != nodata_esperado]
            ).any(),
            "fuera_aoi_es_nodata": np.all(
                datos[mascara_fuera] == nodata_esperado
            )
        }

        validos = (
            (datos != nodata_esperado)
            & np.isfinite(datos)
            & ~mascara_fuera
        )

        checks["hay_datos_validos"] = bool(validos.any())

        return {
            "archivo": Path(ruta).name,
            **checks,
            "todos_correctos": all(checks.values()),
            "pixeles_validos": int(validos.sum())
        }

Exportar archivos

In [ ]:
archivos_salida = sorted(
    CARPETA_SALIDA.glob("*_alineado.tif")
)

if len(archivos_salida) != len(entradas):
    raise RuntimeError(
        "El número de salidas no coincide con el de entradas."
    )

verificaciones = pd.DataFrame([
    verificar_dem(
        ruta,
        transform_grid,
        CRS_TRABAJO,
        ancho,
        alto,
        mascara_fuera_aoi,
        NODATA_SALIDA
    )
    for ruta in archivos_salida
])

display(verificaciones)

assert verificaciones["todos_correctos"].all(), (
    "La verificación espacial falló. "
    "Revisa la tabla antes de continuar."
)

,archivo,crs_correcto,transform_correcto,ancho_correcto,alto_correcto,resolucion_correcta,nodata_correcto,una_banda,dtype_float32,sin_infinitos_validos,fuera_aoi_es_nodata,hay_datos_validos,todos_correctos,pixeles_validos
0,ANADEM_Sinu_alineado.tif,True,True,True,True,True,True,True,True,True,True,True,True,6518932
1,Copernicus_GLO30_Sinu_alineado.tif,True,True,True,True,True,True,True,True,True,True,True,True,6519713
2,FABDEM_Sinu_V12_alineado.tif,True,True,True,True,True,True,True,True,True,True,True,True,6519713


##8. Manifiesto de reproducibilidad

In [ ]:
def sha256_archivo(ruta, bloque=1024 * 1024):
    hash_obj = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        while True:
            datos = archivo.read(bloque)
            if not datos:
                break
            hash_obj.update(datos)

    return hash_obj.hexdigest()


manifest = {
    "proyecto": "Susceptibilidad a inundaciones - Río Sinú",
    "crs": CRS_TRABAJO,
    "resolucion_m": RESOLUCION,
    "origen_superior_izquierdo": [
        transform_grid.c,
        transform_grid.f
    ],
    "transformacion_afín": list(transform_grid)[:6],
    "ancho_pixeles": ancho,
    "alto_pixeles": alto,
    "extension": [
        xmin_grid, ymin_grid, xmax_grid, ymax_grid
    ],
    "nodata_salida": NODATA_SALIDA,
    "remuestreo": "bilinear",
    "mascara": "Centro del píxel dentro del AOI",
    "versiones": VERSIONES,
    "aoi_archivo": RUTA_AOI.name,
    "aoi_capa": NOMBRE_CAPA_AOI,
    "entradas": [
        {
            "archivo": entrada.name,
            "sha256": sha256_archivo(entrada)
        }
        for entrada in entradas
    ],
    "salidas": [
        {
            "archivo": salida.name,
            "sha256": sha256_archivo(salida)
        }
        for salida in archivos_salida
    ]
}

ruta_manifest = CARPETA_SALIDA / "manifest.json"

with open(ruta_manifest, "w", encoding="utf-8") as archivo:
    json.dump(
        manifest,
        archivo,
        ensure_ascii=False,
        indent=2
    )

verificaciones.to_csv(
    CARPETA_SALIDA / "verificacion_espacial.csv",
    index=False
)

print("Manifiesto:", ruta_manifest)
print("Verificación espacial guardada.")

Manifiesto: /content/drive/MyDrive/Sinu_Flood_Susceptibility/02_Data_raw/DEM_alineados/manifest.json
Verificación espacial guardada.
